In [ ]:
# Install LIME, SHAP, and your framework dependencies
!pip -q install lime shap transformers accelerate bitsandbytes spacy
!python -m spacy download en_core_web_sm

# Clone your repo (CHANGE THE URL IF NEEDED)
import os
import sys
if not os.path.exists("hnif_project_test"):
    !git clone https://github.com/your-username/hnif_project_test.git
os.chdir("hnif_project_test")
sys.path.append(os.getcwd())

print("✅ Benchmarking Environment Ready.")

In [ ]:
import time
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_attentions=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# LIME and SHAP require a function that takes a list of strings and returns probabilities
def predict_probabilities(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probs.cpu().numpy()

print("✅ Model and Prediction Wrapper Ready.")

In [ ]:
import lime.lime_text
import shap
from hnif.adapter import run_hnif_analysis

test_text = "The algorithmic progression of artificial intelligence systems demonstrates a remarkable capacity for generating syntactically uniform prose."

print("🏁 STARTING XAI LATENCY BENCHMARK...\n")

# --- 1. LIME BENCHMARK ---
print("▶️ Running LIME (500 perturbations)...")
lime_explainer = lime.lime_text.LimeTextExplainer(class_names=["Class0", "Class1"])
start_time = time.time()
lime_exp = lime_explainer.explain_instance(test_text, predict_probabilities, num_features=10, num_samples=500)
lime_time = time.time() - start_time
print(f"⏱️ LIME Time: {lime_time:.4f} seconds\n")

# --- 2. SHAP BENCHMARK ---
print("▶️ Running SHAP (Partition Explainer)...")
shap_explainer = shap.Explainer(predict_probabilities, tokenizer)
start_time = time.time()
shap_values = shap_explainer([test_text])
shap_time = time.time() - start_time
print(f"⏱️ SHAP Time: {shap_time:.4f} seconds\n")

# --- 3. HNIF BENCHMARK (YOUR FRAMEWORK) ---
print("▶️ Running HNIF (Intrinsic Extraction)...")
# Note: We run it once first to load the LLM into cache so the timing is fair
_ = run_hnif_analysis(model, tokenizer, test_text, "AI-Generated", 98.4)

start_time = time.time()
hnif_results = run_hnif_analysis(model, tokenizer, test_text, "AI-Generated", 98.4)
hnif_time = time.time() - start_time
print(f"⏱️ HNIF Time: {hnif_time:.4f} seconds\n")

# --- RESULTS SUMMARY ---
print("========================================")
print("📊 FINAL LATENCY RESULTS:")
print("========================================")
print(f"LIME: {lime_time:.4f}s")
print(f"SHAP: {shap_time:.4f}s")
print(f"HNIF: {hnif_time:.4f}s (Including LLM Generation!)")
print("========================================")